# 03 Preprocessing

This notebook starts the first preprocessing step after raw BIDS import, manual bad-channel QC, and event handling.

Current goal:

```text
Raw BIDS
  → apply saved bad-channel decisions
  → notch/bandpass filter
  → write filtered raw derivative
```

Output example:

```text
derivatives/meeg-pipeline/sub-0001/meg/preprocessing/
  sub-0001_task-example_desc-filtered_meg.fif
```

This notebook should not contain large processing logic itself. It should call reusable functions from `meeg_pipeline.preprocessing`.


## Setup

In [ ]:
from __future__ import annotations

from pathlib import Path

import pandas as pd

from meeg_pipeline.config import load_config
from meeg_pipeline.channels import summarize_channels, channel_summary_to_dataframe
from meeg_pipeline.preprocessing import (
    make_filtered_raw_path,
    write_filtered_raw_for_recordings,
    load_filtered_raw,
)
from meeg_pipeline.workflow import (
    bad_channels_status_to_dataframe,
    existing_output_policy_for_step,
    find_recording,
    iter_recordings,
    recording_label,
    recording_path_status_to_dataframe,
    selected_recordings_to_dataframe,
    should_overwrite,
)

def find_project_root(start: Path | None = None) -> Path:
    """Find project root by searching upward for configs/local.yaml."""
    start = Path.cwd() if start is None else Path(start).resolve()

    for candidate in [start, *start.parents]:
        if (candidate / "configs" / "local.yaml").exists():
            return candidate

    raise FileNotFoundError(
        "Could not find project root by searching for configs/local.yaml "
        f"above {start}"
    )


PROJECT_ROOT = find_project_root()
CONFIG_PATH = PROJECT_ROOT / "configs" / "local.yaml"
config = load_config(CONFIG_PATH)

print("PROJECT_ROOT:", PROJECT_ROOT)
print("CONFIG_PATH:", CONFIG_PATH)


## Selection

Use single values, lists, `None`, or `"all"`.

Examples:

```python
SUBJECTS = "0001"
SUBJECTS = ["0001", "0002"]
SUBJECTS = "all"

TASKS = "example"
TASKS = ["example"]
TASKS = "all"
TASKS = ["all"]
```

If the project has no sessions or runs, keep `SESSIONS = None` and `RUNS = None`.


In [ ]:
SUBJECTS = "all"
SESSIONS = "all"
TASKS = "all"
RUNS = "all"

# Single-file/manual inspection cells at the end of notebooks are disabled by default
# so batch runs over many participants do not stop for plots or ad-hoc file views.
RUN_SINGLE_FILE_INSPECTIONS = False
selected_recordings = list(
    iter_recordings(
        config,
        subjects=SUBJECTS,
        sessions=SESSIONS,
        tasks=TASKS,
        runs=RUNS,
    )
)

selected_recordings_to_dataframe(selected_recordings)


## Overwrite policy

This notebook writes filtered raw derivatives.

By default, existing outputs are **not** overwritten.

Allowed values:

```python
OVERWRITE_STEPS = []             # overwrite nothing
OVERWRITE_STEPS = ["filtering"]  # overwrite filtered derivatives
OVERWRITE_STEPS = "all"          # overwrite all writing steps
```


In [ ]:
OVERWRITE_STEPS = []

overwrite_settings = pd.DataFrame(
    [
        {
            "step": "filtering",
            "overwrite": should_overwrite("filtering", OVERWRITE_STEPS),
            "policy": existing_output_policy_for_step(
                "filtering",
                OVERWRITE_STEPS,
            ),
        },
    ]
)

overwrite_settings


## Filtering configuration

Filtering parameters are read from `configs/local.yaml`.

Expected config section:

```yaml
preprocessing:
  filtering:
    notch_freqs: [50]
    l_freq: 1.0
    h_freq: 40.0
    method: "fir"
```


In [ ]:
config.preprocessing.filtering


## Check required bad-channel decisions

Filtering applies saved bad-channel decisions before filtering. This table checks whether each selected recording has a corresponding `*_desc-badchannels.json` file.

If a bad-channel decision is missing, go back to the raw-BIDS / bad-channel notebook and complete manual QC first.


In [ ]:
bad_channels_status_to_dataframe(config, selected_recordings)


## Expected filtered outputs

This shows where filtered derivatives will be written.


In [ ]:
recording_path_status_to_dataframe(
    config,
    selected_recordings,
    make_filtered_raw_path,
    exists_column="filtered_exists",
    path_column="output_path",
)


## Write filtered raw derivatives

This step:

1. Loads the raw BIDS recording.
2. Applies saved bad-channel decisions.
3. Applies configured notch and bandpass filters.
4. Writes `desc-filtered_meg.fif` to `derivatives/meeg-pipeline/`.

Existing files are skipped unless `OVERWRITE_STEPS` contains `"filtering"` or is set to `"all"`.


In [ ]:
filter_policy = existing_output_policy_for_step(
    "filtering",
    OVERWRITE_STEPS,
)

filter_results = write_filtered_raw_for_recordings(
    config,
    selected_recordings,
    on_existing=filter_policy,
)

pd.DataFrame(
    [
        {
            "status": result.status,
            "message": result.message,
            "output_path": result.output_path,
        }
        for result in filter_results
    ]
)


## Inspect one filtered output

Use this section to load and inspect one filtered derivative.

You can select a recording explicitly by subject/task/session/run. This is easier than remembering an index while still keeping the batch selection above set to `"all"`.

> Disabled by default for batch runs. Set `RUN_SINGLE_FILE_INSPECTIONS = True` in the selection cell to run this section.



In [ ]:
if RUN_SINGLE_FILE_INSPECTIONS:
    INSPECT_SUBJECT = "0001"
    INSPECT_SESSION = None
    INSPECT_TASK = "example"
    INSPECT_RUN = None

    INSPECT = find_recording(
        selected_recordings,
        subject=INSPECT_SUBJECT,
        session=INSPECT_SESSION,
        task=INSPECT_TASK,
        run=INSPECT_RUN,
    )

    if INSPECT is None:
        filtered_path = None

        inspect_selection_status = pd.DataFrame(
            [
                {
                    "status": "not_selected",
                    "message": (
                        "No matching recording found in selected_recordings. "
                        "Check INSPECT_SUBJECT/SESSION/TASK/RUN or the selection above."
                    ),
                }
            ]
        )

    else:
        filtered_path = make_filtered_raw_path(
            config,
            subject=INSPECT["subject"],
            session=INSPECT["session"],
            task=INSPECT["task"],
            run=INSPECT["run"],
        )

        inspect_selection_status = pd.DataFrame(
            [
                {
                    "recording": recording_label(INSPECT),
                    "filtered_exists": filtered_path.exists(),
                    "filtered_path": str(filtered_path),
                }
            ]
        )

    inspect_selection_status
else:
    print('Skipped single-file inspection cell 16 in 1B_meg_preprocessing/03_preprocessing.ipynb. Set RUN_SINGLE_FILE_INSPECTIONS = True to run it.')


### Load filtered derivative

This uses MNE directly because the filtered file is a derivative, not raw BIDS.

> Disabled by default for batch runs. Set `RUN_SINGLE_FILE_INSPECTIONS = True` in the selection cell to run this section.



In [ ]:
if RUN_SINGLE_FILE_INSPECTIONS:
    if INSPECT is None:
        filtered_result = None
        filtered_raw = None

        pd.DataFrame(
            [
                {
                    "status": "not_selected",
                    "message": (
                        "No matching recording found in selected_recordings. "
                        "Check INSPECT_SUBJECT/SESSION/TASK/RUN or the selection above."
                    ),
                }
            ]
        )

    else:
        filtered_result = load_filtered_raw(
            config,
            subject=INSPECT["subject"],
            session=INSPECT["session"],
            task=INSPECT["task"],
            run=INSPECT["run"],
            preload=False,
        )

        filtered_raw = filtered_result.raw

        pd.DataFrame(
            [
                {
                    "recording": recording_label(INSPECT),
                    "status": filtered_result.status,
                    "message": filtered_result.message,
                    "path": filtered_result.path,
                }
            ]
        )
else:
    print('Skipped single-file inspection cell 18 in 1B_meg_preprocessing/03_preprocessing.ipynb. Set RUN_SINGLE_FILE_INSPECTIONS = True to run it.')


### Channel summary after filtering

Bad channels should still be present in `filtered_raw.info["bads"]`.

> Disabled by default for batch runs. Set `RUN_SINGLE_FILE_INSPECTIONS = True` in the selection cell to run this section.



In [ ]:
if RUN_SINGLE_FILE_INSPECTIONS:
    if filtered_raw is None:
        pd.DataFrame(
            [
                {
                    "status": filtered_result.status if filtered_result is not None else "not_selected",
                    "message": (
                        filtered_result.message
                        if filtered_result is not None
                        else "No matching recording selected."
                    ),
                }
            ]
        )
    else:
        channel_summary = summarize_channels(filtered_raw)
        channel_summary_to_dataframe(channel_summary)
else:
    print('Skipped single-file inspection cell 20 in 1B_meg_preprocessing/03_preprocessing.ipynb. Set RUN_SINGLE_FILE_INSPECTIONS = True to run it.')


### Optional interactive inspection

Run this only when using a local interactive backend.

> Disabled by default for batch runs. Set `RUN_SINGLE_FILE_INSPECTIONS = True` in the selection cell to run this section.



In [ ]:
if RUN_SINGLE_FILE_INSPECTIONS:
    if filtered_raw is None:
        print("No filtered raw loaded. Nothing to plot.")
    else:
        filtered_raw.plot(
            picks="meg",
            block=True,
        )
else:
    print('Skipped single-file inspection cell 22 in 1B_meg_preprocessing/03_preprocessing.ipynb. Set RUN_SINGLE_FILE_INSPECTIONS = True to run it.')
